In [ ]:
# imports

import os
from dotenv import load_dotenv
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


#define prompts
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}]


def summarize(url, model):
    website = fetch_website_contents(url)
    if(model == "openai"):
        load_dotenv(override=True)
        api_key = os.getenv('OPENAI_API_KEY')
        openai = OpenAI()
        response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
        )
    if(model == "gemini"):
        GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
        load_dotenv(override=True)
        google_api_key = os.getenv("GEMINI_API_KEY")
        gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
        response = gemini.chat.completions.create(
        model = "gemini-2.5-flash-lite",
        messages = messages_for(website)
        )
    if(model == "ollama"):
        OLLAMA_BASE_URL = "http://localhost:11434/v1"
        ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')         
        response = ollama.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
        )
    return response.choices[0].message.content


def display_summary(url, model):
    summary = summarize(url, model)
    display(Markdown(summary))

#open AI 
print("Using OPEN AI")
display_summary("https://edwarddonner.com", "openai")

#gemini 
print("Using GEMINI")
display_summary("https://edwarddonner.com", "gemini")

#open AI 
print("Using OLLAMA")
display_summary("https://edwarddonner.com", "ollama")




